# Creating an Agent in Microsoft Foundry

In this quickstart, you use Microsoft Foundry to:

1. Create an agent
2. Chat with an agent

For that, you need to create a Foundry project and deploy a model. If you haven't done that yet, follow the [Get started with code quickstart](https://learn.microsoft.com/en-us/azure/ai-foundry/quickstarts/get-started-code?view=foundry&tabs=python#create-resources) first.

Then you will need to setup environment variables in the `.env` file in this folder. You need to add the `PROJECT_ENDPOINT` variable with the endpoint of your Foundry project. You can find it in the Foundry portal on the welcome screen of your project.

Install these packages, including the preview version of azure-ai-projects.

In [1]:
%pip install azure-ai-projects --pre
%pip install openai azure-identity python-dotenv

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## Chat with a model

Interacting with a model is the basic building block of AI applications. Send an input and receive a response from the model:

In [6]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

load_dotenv()

print(f"Using PROJECT_ENDPOINT: {os.environ['PROJECT_ENDPOINT']}")
print(f"Using MODEL_DEPLOYMENT_NAME: {os.environ['MODEL_DEPLOYMENT_NAME']}")

project_client = AIProjectClient(
    endpoint=os.environ["PROJECT_ENDPOINT"],
    credential=DefaultAzureCredential(),
)

openai_client = project_client.get_openai_client()

response = openai_client.responses.create(
    model=os.environ["MODEL_DEPLOYMENT_NAME"],
    input="What is the size of France in square miles?",
)
print(f"Response output: {response.output_text}")

Using PROJECT_ENDPOINT: https://foundry-270-dev.services.ai.azure.com/api/projects/project-270-dev
Using MODEL_DEPLOYMENT_NAME: gpt-5.2
Response output: France’s total area is about **248,573 square miles** (≈ **643,801 km²**) for **metropolitan France** (mainland + Corsica).  
Including **overseas regions and territories**, it’s about **266,000 square miles** (≈ **688,000 km²**).


## Create an agent

Create an agent using your deployed model.

An agent defines core behavior. Once created, it ensures consistent responses in user interactions without repeating instructions each time. You can update or delete agents anytime.

In [3]:
import os
os.environ["NO_PROXY"] = "*"

In [4]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition

load_dotenv()

project_client = AIProjectClient(
    endpoint=os.environ["PROJECT_ENDPOINT"],
    credential=DefaultAzureCredential(),
)

agent = project_client.agents.create_version(
    agent_name=os.environ["AGENT_NAME"],
    definition=PromptAgentDefinition(
        model=os.environ["MODEL_DEPLOYMENT_NAME"],
        instructions="You are a helpful assistant that answers general questions",
    ),
)
print(f"Agent created (id: {agent.id}, name: {agent.name}, version: {agent.version})")

Agent created (id: MyAgent-202:1, name: MyAgent-202, version: 1)


## Chat with an agent

Use the previously created agent named "MyAgent" to interact by asking a question and a related follow-up. The conversation maintains history across these interactions.

In [5]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

load_dotenv()

project_client = AIProjectClient(
    endpoint=os.environ["PROJECT_ENDPOINT"],
    credential=DefaultAzureCredential(),
)

agent_name = os.environ["AGENT_NAME"]
openai_client = project_client.get_openai_client()

# Optional Step: Create a conversation to use with the agent
conversation = openai_client.conversations.create()
print(f"Created conversation (id: {conversation.id})")

# Chat with the agent to answer questions
response = openai_client.responses.create(
    conversation=conversation.id, #Optional conversation context for multi-turn
    extra_body={"agent": {"name": agent_name, "type": "agent_reference"}},
    input="What is the size of France in square miles?",
)
print(f"Response output: {response.output_text}")

# Optional Step: Ask a follow-up question in the same conversation
response = openai_client.responses.create(
    conversation=conversation.id,
    extra_body={"agent": {"name": agent_name, "type": "agent_reference"}},
    input="And what is the capital city?",
)
print(f"Response output: {response.output_text}")

Created conversation (id: conv_2fbb03da552ffbac00ySk0kzOFJ09twZ7hdioXDLO1zLbYmWPL)
Response output: France covers about **248,600 square miles** in total area (this refers to **metropolitan France**—mainland France plus Corsica).
Response output: The capital city of France is **Paris**.
